# 03 — ClA Estimation from High-Speed Corners

Estimate downforce area (ClA) from Monza's fast corners (Lesmo 1, Lesmo 2, Curva Grande, Parabolica) using the tyre friction model. Validate against published values.

In [1]:
import sys
sys.path.insert(0, '..')

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.segments import extract_corner_samples, _compute_lateral_g_from_position
from src.aero_params import air_density, car_mass, estimate_ClA

G = 9.81
MU_TYRE = 1.8
fastf1.Cache.enable_cache('../cache')

## Load session

In [2]:
session = fastf1.get_session(2024, 'Monza', 'R')
session.load(telemetry=True, weather=True)

winner_abbr = session.results.iloc[0]['Abbreviation']
weather = session.weather_data
rho = air_density(weather['AirTemp'].mean(), weather['Pressure'].mean())
print(f'Driver: {winner_abbr},  ρ = {rho:.4f} kg/m³')

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '4', '55', '44', '1', '63', '11', '23', '20', '14', '43', '3', '31', '10', '77', '27', '24', '18', '22']


Driver: LEC,  ρ = 1.1298 kg/m³


## Check lateral g availability and method

In [ ]:
laps = session.laps.pick_drivers(winner_abbr)
sample_lap = laps.iloc[len(laps)//2]
tel = sample_lap.get_telemetry()

lat_candidates = ['lateral_acceleration', 'LateralAcceleration', 'lateral_g', 'LatG', 'ay']
direct_lat = [c for c in lat_candidates if c in tel.columns]
print('Direct lat-g channels:', direct_lat)
print('GPS position channels:', [c for c in ['X', 'Y'] if c in tel.columns])
METHOD = 'direct' if direct_lat else 'gps'
print('Method:', METHOD)

# ── Diagnostic: what does the computed lat-g actually look like? ──────────
if METHOD == 'gps':
    from src.segments import _compute_lateral_g_from_position
    import pandas as pd
    tel2 = tel.copy().reset_index(drop=True)
    tel2['t'] = tel2['Time'].dt.total_seconds()
    tel2['v_ms'] = tel2['Speed'] / 3.6
    tel2 = _compute_lateral_g_from_position(tel2)

    lat_g = tel2['lat_g_computed'].abs()
    print(f'\nComputed |lat_g| stats:')
    print(f'  max  = {lat_g.max():.2f} g')
    print(f'  p99  = {lat_g.quantile(0.99):.2f} g')
    print(f'  p95  = {lat_g.quantile(0.95):.2f} g')
    print(f'  rows > 1.5g: {(lat_g > 1.5).sum()}')
    print(f'  rows > 2.0g: {(lat_g > 2.0).sum()}')
    print(f'  rows > 2.5g: {(lat_g > 2.5).sum()}')
    print(f'  rows > 3.0g: {(lat_g > 3.0).sum()}')

    # Quick plot to see where lat-g peaks appear on track
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
    axes[0].plot(tel2['Distance'], lat_g)
    axes[0].axhline(2.0, color='r', linestyle='--', label='2.0 g threshold')
    axes[0].set_ylabel('|lat_g| (g)')
    axes[0].set_xlabel('Distance (m)')
    axes[0].set_title(f'GPS-computed lateral g — LEC lap {sample_lap["LapNumber"]}')
    axes[0].legend()

    if 'X' in tel2.columns:
        sc = axes[1].scatter(tel2['X'], tel2['Y'], c=lat_g.clip(0, 4),
                             cmap='hot', s=4)
        plt.colorbar(sc, ax=axes[1], label='|lat_g| (g)')
        axes[1].set_aspect('equal')
        axes[1].set_title('Track coloured by lateral g')

    plt.tight_layout()
    plt.savefig('../results/figures/03_lat_g_diagnostic.png', dpi=150)
    plt.show()

## Collect high-speed corner samples across mid-race laps

In [4]:
# Helper — must be defined before the lap loop below
from src.segments import extract_corner_samples as _ecs

def _collect_samples(tel, m, rho):
    return _ecs(tel, min_speed_kmh=144.0, min_lat_g=3.5, min_throttle=80.0)

In [ ]:
laps_filtered = laps[(laps['LapNumber'] > 2) & (laps['LapNumber'] < laps['LapNumber'].max())]

# Thresholds for GPS-computed lat-g:
# GPS second-derivative values are ~30-40% lower than true IMU values due to
# smoothing and 10 Hz sampling. Monza fast corners peak ~2.5-3.5 g (IMU),
# so GPS-derived peaks are typically 1.5-2.5 g.
LAT_G_THRESH = 1.5
SPEED_THRESH = 130.0   # km/h — Lesmo, Curva Grande, Parabolica
THROTTLE_THRESH = 60.0

ClA_per_lap = []
all_corner_samples = []

for _, lap in laps_filtered.iterrows():
    lap_num = int(lap['LapNumber'])
    m = car_mass(lap_num)
    try:
        tel = lap.get_telemetry()
    except Exception:
        continue

    ClA_med, ClA_std = estimate_ClA(
        tel, m, rho, mu=MU_TYRE,
        min_speed_kmh=SPEED_THRESH,
        min_lat_g=LAT_G_THRESH,
        min_throttle=THROTTLE_THRESH,
    )
    if np.isfinite(ClA_med) and 0.5 < ClA_med < 10.0:
        ClA_per_lap.append({'lap': lap_num, 'ClA': ClA_med, 'ClA_std': ClA_std, 'm': m})

    samples = _collect_samples(tel, m, rho)
    if not samples.empty:
        all_corner_samples.append(samples)

print(f'Laps with valid ClA estimates: {len(ClA_per_lap)}')
if ClA_per_lap:
    vals = [d['ClA'] for d in ClA_per_lap]
    print(f'ClA range: {min(vals):.2f} – {max(vals):.2f} m²  (median {np.median(vals):.2f})')

## ClA vs lap number

In [6]:
if ClA_per_lap:
    df_ClA = pd.DataFrame(ClA_per_lap)
    ClA_global = df_ClA['ClA'].median()
    ClA_global_std = df_ClA['ClA'].std()
    print(f'\nFinal ClA estimate: {ClA_global:.3f} ± {ClA_global_std:.3f} m²')
    print(f'Expected range (low downforce): 2.5 – 3.5 m²')

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.errorbar(df_ClA['lap'], df_ClA['ClA'], yerr=df_ClA['ClA_std'],
                fmt='o', capsize=4, label='per-lap estimate')
    ax.axhline(ClA_global, color='r', linestyle='--', label=f'median = {ClA_global:.3f}')
    ax.fill_between(df_ClA['lap'],
                    ClA_global - ClA_global_std,
                    ClA_global + ClA_global_std,
                    alpha=0.2, color='r')
    ax.set_ylim(0, 6)
    ax.set_xlabel('Lap number')
    ax.set_ylabel('ClA (m²)')
    ax.set_title('Downforce area estimate vs lap')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/figures/03_ClA_vs_lap.png', dpi=150)
    plt.show()
else:
    print('No valid ClA estimates — check lateral g availability and thresholds.')
    

No valid ClA estimates — check lateral g availability and thresholds.


## Lateral g vs v² scatter with tyre friction model

In [7]:
if all_corner_samples and ClA_per_lap:
    combined = pd.concat(all_corner_samples, ignore_index=True)
    v2 = combined['v_ms'].values**2
    lat_a = combined['lat_g'].values * G

    # Tyre model: lat_a = μ*(g + ½ρ*ClA*v²/m)
    # → lat_a/g = μ + (μ*ρ*ClA)/(2*m*g) * v²
    # Slope = μ*ρ*ClA / (2*m*g)  → ClA = slope * 2*m*g / (μ*ρ)
    m_mid = car_mass(25)  # approximate mid-race mass
    v2_range = np.linspace(v2.min(), v2.max(), 200)
    ClA_line = ClA_global
    lat_model = MU_TYRE * (G + 0.5 * rho * ClA_line * v2_range / m_mid)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(v2, lat_a, alpha=0.3, s=10, label='corner samples')
    ax.plot(v2_range, lat_model, 'r-', lw=2,
            label=f'tyre model  ClA={ClA_line:.2f} m²')
    ax.set_xlabel('v² (m²/s²)')
    ax.set_ylabel('lateral acceleration (m/s²)')
    ax.set_title('Lateral g vs v² — tyre friction limit')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/figures/03_lat_g_scatter.png', dpi=150)
    plt.show()

## Save ClA estimate

In [8]:
import pickle
if ClA_per_lap:
    with open('../results/ClA_estimate.pkl', 'wb') as f:
        pickle.dump({'ClA': ClA_global, 'ClA_std': ClA_global_std}, f)
    print('Saved ClA_estimate.pkl')
else:
    print('No ClA estimate to save.')

No ClA estimate to save.
